# IntuBote: training a depth-0 chess transformer on Colab

IntuBote picks a move from the position alone, with no search. It learns from Stockfish evaluations in the Lichess evaluation database.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Run the cells from top to bottom. Data and checkpoints go to Google Drive.
3. **If Colab disconnects,** run all the cells again. The data cell skips work that's already done, and training resumes from the last checkpoint.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/juandanielsg/chess-bote.git"
BRANCH   = "main"                         # the branch that contains intubote/

DRIVE_DIR = "/content/drive/MyDrive/intubote"
POSITIONS = 5_000_000                     # Lichess positions to use (~90 bytes each in RAM)
RUN_NAME  = "intubote_small"

# Model size. "small" ~7M params is a good first run on a T4; "medium" ~26M needs more data/steps.
SIZE = "small"
D_MODEL, LAYERS, HEADS = {"tiny": (128, 4, 4), "small": (256, 8, 8), "medium": (512, 8, 8)}[SIZE]

STEPS  = 100_000                          # optimizer steps (x BATCH positions each)
BATCH  = 1024
LR     = 3e-4
POLICY_TARGET = "best"                    # "best" = copy Stockfish's move, "soft" = spread over top candidates

## 2. Setup

**The repo is private, so Colab needs a read-only GitHub token to clone it** (one-time setup):
1. On GitHub: **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**. Under *Repository access* choose **Only select repositories → chess-bote**, and under *Permissions* set **Contents: Read-only**. Give it an expiry date.
2. In Colab: click the **key icon** in the left sidebar → **Add new secret**, name it `GITHUB_TOKEN`, paste the token, and turn on **Notebook access**.

Colab Secrets belong to your Google account, not to the notebook, so the token isn't saved in the notebook or shared with it.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime -> Change runtime type"

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

%cd /content
!rm -rf chess-bote
# The repo is private: clone with a GitHub token stored in Colab Secrets as GITHUB_TOKEN
# (see the instructions above). It is sent as an HTTP header, so it never shows up in
# the output or in .git/config.
import base64, subprocess
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as e:
    _tok = None
    print(f"No GITHUB_TOKEN secret available ({type(e).__name__}); trying without it.")
_auth = []
if _tok:
    _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    _auth = ["-c", f"http.extraHeader=Authorization: Basic {_b64}"]
_r = subprocess.run(["git", *_auth, "clone", "-q", "--branch", BRANCH, REPO_URL, "chess-bote"],
                    capture_output=True, text=True)
del _tok, _auth
if _r.returncode != 0:
    raise RuntimeError("git clone failed. Check the GITHUB_TOKEN secret (and its notebook "
                       "access toggle) and that the token can read the repo.\n" + _r.stderr)
print("cloned", REPO_URL, "@", BRANCH)
%cd /content/chess-bote
# IntuBote only needs chess-bote's pure-Python board (for legal moves), so no C++ build is needed.
os.environ["PYTHONPATH"] = "/content/chess-bote/src"
!pip install -q zstandard orjson
!test -f intubote/train.py && echo "intubote/ found" || echo "ERROR: branch has no intubote/ (push it first)"

## 3. Data: stream positions from the Lichess evaluation database

This reads only the first `POSITIONS` entries of the multi-GB file and never downloads the whole thing. It runs at roughly 10k positions/s, so 5M takes about 10 minutes. The output is saved to Drive and reused in later sessions.

In [ ]:
DATA_DIR = f"{DRIVE_DIR}/data_{POSITIONS // 1_000_000}M"
if os.path.exists(f"{DATA_DIR}/meta.json"):
    print("data already prepared:", open(f"{DATA_DIR}/meta.json").read())
else:
    !python intubote/prepare_data.py --count {POSITIONS} --out /content/data_tmp
    !mkdir -p {DATA_DIR} && cp /content/data_tmp/* {DATA_DIR}/

# Train from local disk; Drive is slow to read.
!mkdir -p /content/data && cp -n {DATA_DIR}/* /content/data/

## 4. Train (with a live progress chart)

The chart redraws after every evaluation (every 1,000 steps):
- **Move accuracy:** how often the net's first choice (top-1), or one of its first three (top-3), is Stockfish's best move. Higher is better. Use it to compare runs, but judge strength by playing games.
- **Policy / value loss:** training vs validation. If validation flattens or rises while training keeps falling, the model is overfitting and needs more data.

Checkpoints are saved to Drive every 2,000 steps. Stopping the cell, or a disconnect, loses at most that much work, and rerunning the cell resumes from the last checkpoint with the chart history intact.

In [ ]:
import io, subprocess, sys
import matplotlib.pyplot as plt
from IPython.display import Image, display
sys.path.insert(0, "/content/chess-bote/intubote")
from plot_metrics import plot_metrics

CKPT = f"{DRIVE_DIR}/{RUN_NAME}.pt"
METRICS = CKPT + ".metrics.jsonl"

def draw():
    fig = plot_metrics(METRICS)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    return Image(buf.getvalue())

chart = display(draw() if os.path.exists(METRICS) else "waiting for the first evaluation...",
                display_id=True)
log = display("", display_id=True)

cmd = ["python", "-u", "intubote/train.py", "--data", "/content/data", "--out", CKPT,
       "--steps", str(STEPS), "--batch", str(BATCH), "--lr", str(LR),
       "--policy-target", POLICY_TARGET,
       "--d-model", str(D_MODEL), "--layers", str(LAYERS), "--heads", str(HEADS)]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
# Setup/diagnostic lines stay pinned at the top; below them, the latest progress lines.
PINNED = ("[check]", "device:", "data:", "model:", "gpu:", "resuming", "training from")
header, tail, everything = [], [], []
try:
    for line in proc.stdout:
        line = line.rstrip()
        everything.append(line)
        if line.startswith(PINNED):
            header.append(line)
        else:
            tail = (tail + [line])[-10:]
        log.update({"text/plain": "\n".join(header + (["..."] + tail if tail else []))}, raw=True)
        if line.startswith("step "):
            chart.update(draw())
    proc.wait()
finally:
    if proc.poll() is None:          # cell stopped by hand: stop training too
        proc.terminate()
        proc.wait()
print("exit code", proc.returncode)
if proc.returncode not in (0, None, -15):   # crashed: show the full error, not just the tail
    print("\n".join(everything[-40:]))

### Progress chart (any time)

Redraws the chart from the log on Drive. Use it after a reconnect, or to look at a past run by changing `RUN_NAME` and rerunning the cell above it first.

In [ ]:
display(draw())

## 5. Try it

In [ ]:
for fen in ["rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
            "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4",   # Qxf7#
            "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR b KQkq - 3 3"]:  # stop Qxf7#
    print(fen)
    !python intubote/engine.py --ckpt {CKPT} --fen "{fen}" --top 3

## 6. Play it on your machine

Download `RUN_NAME.pt` from your Drive folder, then in the repo:

```bash
uv sync --group intubote
uv run python intubote/uci.py --ckpt path/to/intubote_small.pt     # add this command as a UCI engine in CuteChess/Arena
```

To measure strength, play it against chess-bote or Stockfish at limited Elo with `cutechess-cli`, like `scripts/match.sh` does.